## Persistent Landing - Delta Lake

The aim of this notebook is to convert all structure data we have into parquet files, in our case, it would be CSV files.

**Importing Useful Libraries**

In [1]:
import os
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [4]:
# Create a specilized sub-bucket to store parquet files inside persistent-landing
s3.put_object(Bucket="landing-zone", Key="persistent-landing/csv-delta-lake/")

{'ResponseMetadata': {'RequestId': '189D99E26B7F0938',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': 'AAAAAA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '189D99E26B7F0938',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '9099',
   'x-ratelimit-remaining': '9099',
   'x-xss-protection': '1; mode=block',
   'date': 'Tue, 17 Mar 2026 10:23:46 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
 'ChecksumCRC32': 'AAAAAA==',
 'ChecksumType': 'FULL_OBJECT'}

To build a high-performance "**Bronze Layer**" (this is simulated by the sub-bucket `csv-delta-lake`), this pipeline automates the conversion of raw CSV files into versioned **Delta Tables** by leveraging **DuckDB** for high-speed S3 streaming and Parquet conversion, followed by **Polars** to finalize the ACID-compliant Delta format. This transition optimizes the data lake for columnar performance and transactional integrity (via the `_delta_log`), while maintaining a clean environment by automatically purging intermediate Parquet files and providing full compatibility with S3-compatible storage through specialized `storage_options`.

In [5]:
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

def ingest_csv_to_delta(bucket, csv_prefix="persistent-landing/csv/"):
    """
    Unified pipeline: 
    1. DuckDB reads CSV and converts to a temporary Parquet.
    2. Polars/DeltaLake reads that Parquet and writes it as a versioned Delta Table.
    3. Cleans up the temporary Parquet.
    """
    # Initialize DuckDB S3 access
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    paginator = s3.get_paginator("list_objects_v2")
    delta_base_prefix = "persistent-landing/csv-delta-lake"

    for page in paginator.paginate(Bucket=bucket, Prefix=csv_prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]
            
            # Skip directories
            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # Extract base name (e.g., 'users' from 'path/to/users.csv')
            file_base = os.path.splitext(os.path.basename(src_key))[0]
            
            # Path Definitions
            s3_csv_path = f"s3://{bucket}/{src_key}"
            temp_parquet_path = f"s3://{bucket}/{delta_base_prefix}/{file_base}_temp.parquet"
            target_delta_folder = f"s3://{bucket}/{delta_base_prefix}/{file_base}/"

            print(f"🚀 Processing: {file_base}...")

            try:
                # --- Phase 1: DuckDB (CSV to Parquet) ---
                con.execute(f"""
                    COPY (SELECT * FROM read_csv_auto('{s3_csv_path}')) 
                    TO '{temp_parquet_path}' (FORMAT PARQUET);
                """)
                print(f"  └─ DuckDB: CSV converted to temporary Parquet.")

                # --- Phase 2: Delta Lake (Parquet to Delta) ---
                df = pl.read_parquet(temp_parquet_path, storage_options=storage_options)
                
                write_deltalake(
                    target_delta_folder,
                    df,
                    mode="overwrite",
                    storage_options=storage_options
                )
                print(f"  └─ Delta: Version 0 created at {target_delta_folder}")

                # --- Phase 3: Cleanup ---
                # Delete the temporary parquet file
                s3.delete_object(Bucket=bucket, Key=f"{delta_base_prefix}/{file_base}_temp.parquet")
                print(f"✅ Successfully finalized {file_base}.")

            except Exception as e:
                print(f"❌ Failed to process {file_base}: {e}")

In [6]:
# Convert csv files into parquet files
ingest_csv_to_delta("landing-zone", "persistent-landing/csv/")

🚀 Processing: co2-emission-by-vehicles_1773742652610...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 created at s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission-by-vehicles_1773742652610/
✅ Successfully finalized co2-emission-by-vehicles_1773742652610.
🚀 Processing: global_warming_dataset_1773742652711...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 created at s3://landing-zone/persistent-landing/csv-delta-lake/global_warming_dataset_1773742652711/
✅ Successfully finalized global_warming_dataset_1773742652711.
🚀 Processing: natural_disaster_tweets_1773743011182...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 created at s3://landing-zone/persistent-landing/csv-delta-lake/natural_disaster_tweets_1773743011182/
✅ Successfully finalized natural_disaster_tweets_1773743011182.
🚀 Processing: temperature_change_1773743011379...
  └─ DuckDB: CSV converted to temporary Parquet.
  └─ Delta: Version 0 cre

We can now try querying over these parquet filed.

In [7]:
print("🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:")
table_path = "s3://landing-zone/persistent-landing/csv-delta-lake/co2-emission*/part*.parquet"
df_view = con.execute(f"SELECT * FROM read_parquet('{table_path}') LIMIT 10").df()
df_view

🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:


,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244
5,ACURA,RLX,MID-SIZE,3.5,6,AS6,Z,11.9,7.7,10.0,28,230
6,ACURA,TL,MID-SIZE,3.5,6,AS6,Z,11.8,8.1,10.1,28,232
7,ACURA,TL AWD,MID-SIZE,3.7,6,AS6,Z,12.8,9.0,11.1,25,255
8,ACURA,TL AWD,MID-SIZE,3.7,6,M6,Z,13.4,9.5,11.6,24,267
9,ACURA,TSX,COMPACT,2.4,4,AS5,Z,10.6,7.5,9.2,31,212
